# 🤖 Scikit-learn — Полный обучающий блокнот

**Для студентов 1 курса ИИИ ИТМО**

---

## Содержание

### Часть I — Основы sklearn
1. [Установка и импорт](#1)
2. [Философия sklearn: Estimator API](#2)
3. [Встроенные датасеты](#3)
4. [Train/test split и кросс-валидация](#4)

### Часть II — Preprocessing (предобработка)
5. [Масштабирование: StandardScaler, MinMaxScaler, RobustScaler](#5)
6. [Кодирование категорий: OrdinalEncoder, OneHotEncoder, LabelEncoder](#6)
7. [Обработка пропусков: SimpleImputer, KNNImputer](#7)
8. [Трансформация признаков: PolynomialFeatures, FunctionTransformer, PowerTransformer](#8)
9. [Отбор признаков: SelectKBest, SelectFromModel, RFE](#9)

### Часть III — Модели
10. [Линейная регрессия (LinearRegression, Ridge, Lasso, ElasticNet)](#10)
11. [Логистическая регрессия](#11)
12. [KNN (K ближайших соседей)](#12)
13. [Деревья решений и случайный лес](#13)
14. [Gradient Boosting (GradientBoostingClassifier/Regressor)](#14)
15. [SVM (Support Vector Machines)](#15)

### Часть IV — Метрики
16. [Метрики классификации: accuracy, precision, recall, f1, ROC AUC, confusion matrix](#16)
17. [Метрики регрессии: MSE, RMSE, MAE, R², MAPE](#17)

### Часть V — Pipelines
18. [Pipeline: зачем и как](#18)
19. [ColumnTransformer: разные обработки для разных столбцов](#19)
20. [Полный Pipeline: preprocessing + model](#20)
21. [make_pipeline и make_column_transformer](#21)

### Часть VI — Подбор гиперпараметров
22. [GridSearchCV](#22)
23. [RandomizedSearchCV](#23)
24. [Подбор по Pipeline](#24)

### Часть VII — Продвинутое
25. [Кастомные трансформеры](#25)
26. [Сохранение и загрузка моделей (joblib, pickle)](#26)
27. [Полный пример: от сырых данных до модели](#27)
28. [Шпаргалка](#28)

---

### Полезные ссылки

- [Scikit-learn Documentation](https://scikit-learn.org/stable/)
- [User Guide](https://scikit-learn.org/stable/user_guide.html)
- [API Reference](https://scikit-learn.org/stable/modules/classes.html)
- [Tutorials](https://scikit-learn.org/stable/tutorial/index.html)
- [Choosing the right estimator (flowchart)](https://scikit-learn.org/stable/machine_learning_map.html)
- [Pipeline User Guide](https://scikit-learn.org/stable/modules/compose.html)
- [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)
- [Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)

---
# Часть I — Основы sklearn
---

<a id='1'></a>
## 1. Установка и импорт

In [ ]:
# !pip install scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print(f'scikit-learn version: {sklearn.__version__}')

# Настройки
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', font_scale=1.0)

import warnings
warnings.filterwarnings('ignore')

print('Готово! ✅')

<a id='2'></a>
## 2. Философия sklearn: Estimator API

Все объекты sklearn следуют **единому интерфейсу**:

```
Transformers (преобразователи):         Predictors (предсказатели):
  .fit(X)         — обучиться             .fit(X, y)     — обучиться
  .transform(X)   — преобразовать         .predict(X)    — предсказать
  .fit_transform(X) — обучиться           .score(X, y)   — оценить
                     + преобразовать       .predict_proba(X) — вероятности
```

**Золотое правило:** `.fit()` вызывается ТОЛЬКО на обучающих данных!

Документация: [Estimator API](https://scikit-learn.org/stable/developers/develop.html)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# === Transformer ===
scaler = StandardScaler()

X_train = np.array([[1, 100], [2, 200], [3, 300], [4, 400]])
X_test  = np.array([[5, 500], [6, 600]])

scaler.fit(X_train)                   # вычислить mean и std на train
X_train_scaled = scaler.transform(X_train)  # применить
X_test_scaled  = scaler.transform(X_test)   # те же mean/std!

print('mean (learned):', scaler.mean_)
print('std (learned): ', scaler.scale_)
print('X_train_scaled:\n', X_train_scaled)
print('X_test_scaled:\n', X_test_scaled)

# fit_transform = fit + transform
X_train_scaled2 = scaler.fit_transform(X_train)
print('\nfit_transform == fit+transform:', np.allclose(X_train_scaled, X_train_scaled2))

In [ ]:
# === Predictor ===
model = LinearRegression()

X = np.array([[1], [2], [3], [4], [5]])
y = np.array([2.1, 3.9, 6.2, 7.8, 10.1])

model.fit(X, y)                   # обучить
y_pred = model.predict(X)         # предсказать
score = model.score(X, y)         # R² score

print(f'coef: {model.coef_[0]:.4f}')
print(f'intercept: {model.intercept_:.4f}')
print(f'R²: {score:.4f}')
print(f'Predictions: {y_pred.round(2)}')

<a id='3'></a>
## 3. Встроенные датасеты

Документация: [Datasets](https://scikit-learn.org/stable/datasets.html)

In [ ]:
from sklearn.datasets import (
    load_iris,              # классификация (3 класса)
    load_wine,              # классификация (3 класса)
    load_breast_cancer,     # бинарная классификация
    load_diabetes,          # регрессия
    fetch_california_housing,  # регрессия (загрузка из интернета)
    make_classification,    # генерация данных
    make_regression,
    make_blobs,
)

# Формат Bunch
iris = load_iris()
print('Тип:', type(iris))
print('Ключи:', list(iris.keys()))
print(f'X: {iris.data.shape}, y: {iris.target.shape}')
print(f'Features: {iris.feature_names}')
print(f'Classes: {iris.target_names}')

# В DataFrame
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['target'] = iris.target
df_iris['species'] = df_iris['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})
print('\n', df_iris.head())

In [ ]:
# Генерация синтетических данных
X_cls, y_cls = make_classification(
    n_samples=500, n_features=10, n_informative=5,
    n_redundant=2, n_classes=2, random_state=42
)
print(f'Classification: X={X_cls.shape}, y={y_cls.shape}')

X_reg, y_reg = make_regression(
    n_samples=500, n_features=5, n_informative=3,
    noise=10, random_state=42
)
print(f'Regression: X={X_reg.shape}, y={y_reg.shape}')

<a id='4'></a>
## 4. Train/test split и кросс-валидация

**Никогда** не оценивайте модель на данных, на которых она обучалась!

Документация: [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)

In [ ]:
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold,
    StratifiedKFold,
    LeaveOneOut,
)

X, y = load_iris(return_X_y=True)

# === train_test_split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,         # 20% на тест
    random_state=42,       # воспроизводимость
    stratify=y,            # сохранить пропорции классов!
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train classes: {np.bincount(y_train)}')
print(f'Test classes:  {np.bincount(y_test)}')

In [ ]:
# === Кросс-валидация ===
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200)

# Простая K-fold CV
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f'5-Fold CV scores: {scores}')
print(f'Mean: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# === Разные стратегии разбиения ===

# KFold — обычное разбиение
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kf = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
print(f'KFold:          {scores_kf.mean():.4f} ± {scores_kf.std():.4f}')

# StratifiedKFold — сохраняет пропорции классов (для классификации!)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_skf = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
print(f'StratifiedKFold: {scores_skf.mean():.4f} ± {scores_skf.std():.4f}')

In [ ]:
# === cross_validate — больше информации ===
from sklearn.model_selection import cross_validate

cv_results = cross_validate(
    model, X, y, cv=5,
    scoring=['accuracy', 'f1_macro'],
    return_train_score=True,
)

print('Ключи:', list(cv_results.keys()))
print(f'Test accuracy:  {cv_results["test_accuracy"].mean():.4f}')
print(f'Train accuracy: {cv_results["train_accuracy"].mean():.4f}')
print(f'Test F1 macro:  {cv_results["test_f1_macro"].mean():.4f}')

---
# Часть II — Preprocessing
---

<a id='5'></a>
## 5. Масштабирование

Многие алгоритмы (KNN, SVM, LogReg, нейросети) чувствительны к масштабу признаков.

Документация: [Preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

np.random.seed(42)
X = np.column_stack([
    np.random.normal(100, 20, 200),    # разные масштабы
    np.random.exponential(5, 200),
    np.random.uniform(0, 1, 200),
])

scalers = {
    'StandardScaler': StandardScaler(),    # (x - mean) / std → mean=0, std=1
    'MinMaxScaler': MinMaxScaler(),        # (x - min) / (max - min) → [0, 1]
    'RobustScaler': RobustScaler(),        # (x - median) / IQR — устойчив к выбросам
    'MaxAbsScaler': MaxAbsScaler(),        # x / max(|x|) → [-1, 1]
}

fig, axes = plt.subplots(1, len(scalers) + 1, figsize=(20, 4))

axes[0].boxplot(X)
axes[0].set_title('Original')
axes[0].set_xticklabels(['F1', 'F2', 'F3'])

for i, (name, scaler) in enumerate(scalers.items(), 1):
    X_scaled = scaler.fit_transform(X)
    axes[i].boxplot(X_scaled)
    axes[i].set_title(name, fontsize=10)
    axes[i].set_xticklabels(['F1', 'F2', 'F3'])

fig.suptitle('Сравнение скейлеров', fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# === Правильный порядок: fit на train, transform на test ===
X_train, X_test = X[:150], X[150:]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform
X_test_s  = scaler.transform(X_test)        # только transform!

print(f'Train mean: {X_train_s.mean(axis=0).round(6)}, std: {X_train_s.std(axis=0).round(4)}')
print(f'Test mean:  {X_test_s.mean(axis=0).round(4)}, std: {X_test_s.std(axis=0).round(4)}')
print('Test != 0/1 — это нормально! Параметры из train.')

<a id='6'></a>
## 6. Кодирование категорий

Документация: [Encoding categorical features](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder

df = pd.DataFrame({
    'color': ['red', 'blue', 'green', 'red', 'blue'],
    'size': ['S', 'M', 'L', 'XL', 'M'],
    'price': [10, 20, 30, 15, 25],
})
print('Исходные данные:')
print(df, '\n')

In [ ]:
# === OrdinalEncoder — числа 0, 1, 2... ===
# Для признаков с ЕСТЕСТВЕННЫМ порядком (size: S < M < L < XL)

oe = OrdinalEncoder(categories=[['S', 'M', 'L', 'XL']])  # порядок важен!
size_encoded = oe.fit_transform(df[['size']])
print('OrdinalEncoder (size):')
print(size_encoded.flatten())  # [0, 1, 2, 3, 1]

In [ ]:
# === OneHotEncoder — бинарные столбцы ===
# Для НОМИНАЛЬНЫХ признаков (color: нет порядка)

ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' — избежать мультиколлинеарности
color_ohe = ohe.fit_transform(df[['color']])
print('OneHotEncoder (color):')
print(f'Колонки: {ohe.get_feature_names_out()}')
print(color_ohe)

In [ ]:
# === LabelEncoder — для ЦЕЛЕВОЙ переменной ===
le = LabelEncoder()
y = ['cat', 'dog', 'cat', 'bird', 'dog']
y_enc = le.fit_transform(y)
print(f'LabelEncoder: {y} -> {y_enc}')
print(f'Обратно: {le.inverse_transform(y_enc)}')
print(f'Классы: {le.classes_}')

<a id='7'></a>
## 7. Обработка пропусков

Документация: [Imputation](https://scikit-learn.org/stable/modules/impute.html)

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

X = np.array([
    [1, 2, np.nan],
    [3, np.nan, 6],
    [7, 8, 9],
    [np.nan, 11, 12],
    [4, 5, 6],
])
print('С пропусками:\n', X, '\n')

# Среднее
imp_mean = SimpleImputer(strategy='mean')
print('mean:\n', imp_mean.fit_transform(X), '\n')

# Медиана
imp_median = SimpleImputer(strategy='median')
print('median:\n', imp_median.fit_transform(X), '\n')

# Константа
imp_const = SimpleImputer(strategy='constant', fill_value=-999)
print('constant (-999):\n', imp_const.fit_transform(X), '\n')

# KNN Imputer — заполняет на основе соседей
imp_knn = KNNImputer(n_neighbors=2)
print('KNN (k=2):\n', imp_knn.fit_transform(X))

In [ ]:
# SimpleImputer для строк
X_str = np.array([['red', 'big'],
                   [None, 'small'],
                   ['blue', None],
                   ['red', 'small']])

imp_freq = SimpleImputer(strategy='most_frequent')  # мода
print('most_frequent:\n', imp_freq.fit_transform(X_str))

<a id='8'></a>
## 8. Трансформация признаков

Документация: [Feature transformations](https://scikit-learn.org/stable/modules/preprocessing.html#non-linear-transformation)

In [ ]:
from sklearn.preprocessing import (
    PolynomialFeatures,
    FunctionTransformer,
    PowerTransformer,
    Binarizer,
)

# === PolynomialFeatures — создание полиномиальных признаков ===
X = np.array([[2, 3], [4, 5]])
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly = poly.fit_transform(X)
print('PolynomialFeatures (degree=2):')
print(f'Имена: {poly.get_feature_names_out()}')
print(X_poly, '\n')

# Только взаимодействия
poly_int = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
print('interaction_only:')
print(f'Имена: {poly_int.fit_transform(X)}')

In [ ]:
# === FunctionTransformer — обёртка для произвольной функции ===
log_transformer = FunctionTransformer(np.log1p, validate=True)

X = np.array([[1, 100], [10, 1000], [100, 10000]])
print('log1p:', log_transformer.fit_transform(X).round(3))

In [ ]:
# === PowerTransformer — приведение к нормальному распределению ===
np.random.seed(42)
X_skewed = np.random.exponential(5, (200, 1))

pt = PowerTransformer(method='yeo-johnson')  # или 'box-cox' (только для >0)
X_normal = pt.fit_transform(X_skewed)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(X_skewed, bins=30, color='coral', edgecolor='white')
axes[0].set_title(f'До (skew={pd.Series(X_skewed.flatten()).skew():.2f})')
axes[1].hist(X_normal, bins=30, color='steelblue', edgecolor='white')
axes[1].set_title(f'После (skew={pd.Series(X_normal.flatten()).skew():.2f})')
fig.suptitle('PowerTransformer (Yeo-Johnson)', fontsize=13)
fig.tight_layout()
plt.show()

<a id='9'></a>
## 9. Отбор признаков

Документация: [Feature selection](https://scikit-learn.org/stable/modules/feature_selection.html)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.ensemble import RandomForestClassifier

X, y = load_iris(return_X_y=True)
feature_names = load_iris().feature_names

# === SelectKBest — топ-K по статистическому тесту ===
selector = SelectKBest(score_func=f_classif, k=2)
X_selected = selector.fit_transform(X, y)

print('SelectKBest (k=2):')
print(f'Scores: {dict(zip(feature_names, selector.scores_.round(2)))}')
mask = selector.get_support()
print(f'Выбраны: {np.array(feature_names)[mask]}')
print(f'Форма: {X.shape} -> {X_selected.shape}')

In [ ]:
# === SelectFromModel — по важности в модели ===
rf = RandomForestClassifier(n_estimators=100, random_state=42)
sfm = SelectFromModel(rf, threshold='median')  # оставить выше медианы важности
X_sfm = sfm.fit_transform(X, y)

importances = sfm.estimator_.feature_importances_
print('\nSelectFromModel (RF):')
print(f'Importances: {dict(zip(feature_names, importances.round(3)))}')
print(f'Выбраны: {np.array(feature_names)[sfm.get_support()]}')

In [ ]:
# === RFE (Recursive Feature Elimination) ===
from sklearn.linear_model import LogisticRegression

rfe = RFE(estimator=LogisticRegression(max_iter=200), n_features_to_select=2)
rfe.fit(X, y)

print('RFE:')
print(f'Ранги: {dict(zip(feature_names, rfe.ranking_))}')
print(f'Выбраны: {np.array(feature_names)[rfe.support_]}')

---
# Часть III — Модели
---

<a id='10'></a>
## 10. Линейная регрессия

Документация: [Linear Models](https://scikit-learn.org/stable/modules/linear_model.html)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

X, y = make_regression(n_samples=200, n_features=5, n_informative=3, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'LinearRegression': LinearRegression(),
    'Ridge (α=1)': Ridge(alpha=1.0),            # L2 регуляризация
    'Lasso (α=1)': Lasso(alpha=1.0),            # L1 регуляризация (обнуляет коэффициенты!)
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5),  # L1 + L2
}

print(f'{"Model":25s} {"Train R²":>10s} {"Test R²":>10s} {"Test RMSE":>10s}')
print('-' * 60)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    train_r2 = model.score(X_train, y_train)
    test_r2 = r2_score(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    print(f'{name:25s} {train_r2:10.4f} {test_r2:10.4f} {rmse:10.2f}')

In [ ]:
# Визуализация коэффициентов
fig, ax = plt.subplots(figsize=(10, 4))
coef_df = pd.DataFrame({
    name: model.coef_ for name, model in models.items()
}, index=[f'x{i}' for i in range(X.shape[1])])

coef_df.plot(kind='bar', ax=ax)
ax.set_title('Коэффициенты моделей')
ax.set_ylabel('Коэффициент')
ax.legend(bbox_to_anchor=(1.05, 1))
ax.axhline(0, color='black', linewidth=0.5)
fig.tight_layout()
plt.show()

<a id='11'></a>
## 11. Логистическая регрессия

Несмотря на название — это модель **классификации**.

Документация: [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

In [ ]:
from sklearn.linear_model import LogisticRegression

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Масштабирование важно для LogReg!
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

log_reg = LogisticRegression(
    C=1.0,                  # обратная сила регуляризации (больше = слабее)
    penalty='l2',            # 'l1', 'l2', 'elasticnet', None
    solver='lbfgs',          # 'liblinear', 'lbfgs', 'saga'
    max_iter=200,
    multi_class='multinomial',  # для >2 классов
)

log_reg.fit(X_train_s, y_train)

print(f'Train accuracy: {log_reg.score(X_train_s, y_train):.4f}')
print(f'Test accuracy:  {log_reg.score(X_test_s, y_test):.4f}')

# Вероятности
proba = log_reg.predict_proba(X_test_s[:3])
print(f'\nВероятности (первые 3):\n{proba.round(3)}')
print(f'Классы: {log_reg.classes_}')

<a id='12'></a>
## 12. KNN (K ближайших соседей)

Документация: [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Подбор K
k_values = range(1, 21)
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    train_scores.append(knn.score(X_train_s, y_train))
    test_scores.append(knn.score(X_test_s, y_test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, train_scores, 'o-', label='Train')
ax.plot(k_values, test_scores, 's-', label='Test')
ax.set_xlabel('K (число соседей)')
ax.set_ylabel('Accuracy')
ax.set_title('KNN: выбор K')
ax.legend()
ax.grid(True, alpha=0.3)
best_k = k_values[np.argmax(test_scores)]
ax.axvline(best_k, color='red', ls='--', alpha=0.5, label=f'Best K={best_k}')
ax.legend()
fig.tight_layout()
plt.show()

print(f'Best K={best_k}, Test accuracy={max(test_scores):.4f}')

<a id='13'></a>
## 13. Деревья решений и случайный лес

Документация:
- [Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- [Random Forest](https://scikit-learn.org/stable/modules/ensemble.html#random-forests)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Decision Tree
dt = DecisionTreeClassifier(
    max_depth=3,              # глубина
    min_samples_split=5,      # мин. образцов для разбиения
    min_samples_leaf=2,       # мин. образцов в листе
    random_state=42,
)
dt.fit(X_train, y_train)
print(f'Decision Tree: train={dt.score(X_train, y_train):.4f}, test={dt.score(X_test, y_test):.4f}')

# Random Forest
rf = RandomForestClassifier(
    n_estimators=100,         # число деревьев
    max_depth=5,
    min_samples_split=5,
    max_features='sqrt',      # признаков на разбиение
    random_state=42,
    n_jobs=-1,                # все ядра
)
rf.fit(X_train, y_train)
print(f'Random Forest: train={rf.score(X_train, y_train):.4f}, test={rf.score(X_test, y_test):.4f}')

In [ ]:
# Feature importance
feature_names = load_iris().feature_names
importances = rf.feature_importances_
idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(np.array(feature_names)[idx], importances[idx], color='steelblue')
ax.set_title('Feature Importance (Random Forest)')
ax.set_xlabel('Importance')
fig.tight_layout()
plt.show()

In [ ]:
# Текстовое представление дерева
print(export_text(dt, feature_names=list(feature_names), max_depth=3))

<a id='14'></a>
## 14. Gradient Boosting

Документация: [Gradient Boosting](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingClassifier  # быстрый вариант

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,        # скорость обучения
    max_depth=3,
    subsample=0.8,            # доля данных на каждое дерево
    random_state=42,
)
gb.fit(X_train, y_train)
print(f'GradientBoosting: train={gb.score(X_train, y_train):.4f}, test={gb.score(X_test, y_test):.4f}')

# HistGradientBoosting — быстрее, поддерживает NaN
hgb = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth=3, random_state=42)
hgb.fit(X_train, y_train)
print(f'HistGBT:          train={hgb.score(X_train, y_train):.4f}, test={hgb.score(X_test, y_test):.4f}')

<a id='15'></a>
## 15. SVM (Support Vector Machines)

Документация: [SVM](https://scikit-learn.org/stable/modules/svm.html)

In [ ]:
from sklearn.svm import SVC, SVR, LinearSVC

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# SVM ОБЯЗАТЕЛЬНО масштабировать!
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

kernels = ['linear', 'rbf', 'poly']
for kernel in kernels:
    svm = SVC(kernel=kernel, C=1.0, random_state=42)
    svm.fit(X_train_s, y_train)
    print(f'SVM ({kernel:6s}): train={svm.score(X_train_s, y_train):.4f}, '
          f'test={svm.score(X_test_s, y_test):.4f}')

---
# Часть IV — Метрики
---

<a id='16'></a>
## 16. Метрики классификации

Документация: [Classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
)

# Бинарная классификация
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=200)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]  # вероятность класса 1

print('=== Базовые метрики ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1:        {f1_score(y_test, y_pred):.4f}')
print(f'ROC AUC:   {roc_auc_score(y_test, y_proba):.4f}')

In [ ]:
# === Classification Report (всё сразу) ===
print(classification_report(y_test, y_pred, target_names=['malignant', 'benign']))

In [ ]:
# === Confusion Matrix ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Вариант 1: ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['malignant', 'benign'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion Matrix')

# Вариант 2: нормализованная
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['malignant', 'benign'],
    normalize='true',  # по строкам (true labels)
    cmap='Blues', ax=axes[1],
    values_format='.2%'
)
axes[1].set_title('Normalized Confusion Matrix')

fig.tight_layout()
plt.show()

In [ ]:
# === ROC Curve ===
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
axes[0].plot(fpr, tpr, linewidth=2, label=f'ROC (AUC={auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall
precision, recall, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

axes[1].plot(recall, precision, linewidth=2, label=f'PR (AP={ap:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

<a id='17'></a>
## 17. Метрики регрессии

Документация: [Regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)

In [ ]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
    median_absolute_error,
)

X, y = make_regression(n_samples=200, n_features=5, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse  = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)  # или np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
medae = median_absolute_error(y_test, y_pred)

print(f'MSE:  {mse:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'MAE:  {mae:.2f}')
print(f'MedAE:{medae:.2f}')
print(f'R²:   {r2:.4f}')
print(f'MAPE: {mape:.4f} ({mape*100:.1f}%)')

In [ ]:
# === Визуализация: predicted vs actual ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(y_test, y_pred, alpha=0.6)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', label='Ideal')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Predicted vs Actual (R²={r2:.3f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.6)
axes[1].axhline(0, color='red', ls='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals plot')
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
# Часть V — Pipelines
---

<a id='18'></a>
## 18. Pipeline: зачем и как

**Pipeline** — цепочка шагов (transformer → ... → transformer → estimator).

Зачем:
1. **Нет утечки данных** — fit только на train
2. **Удобство** — один объект вместо десяти
3. **Воспроизводимость** — легко сохранить/загрузить
4. **GridSearch** — можно искать по параметрам всех шагов

Документация: [Pipeline](https://scikit-learn.org/stable/modules/compose.html#pipeline)

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# === Без Pipeline (опасно!) ===
# scaler = StandardScaler()
# X_train_s = scaler.fit_transform(X_train)
# X_test_s = scaler.transform(X_test)     # легко забыть!
# model = LogisticRegression()
# model.fit(X_train_s, y_train)
# model.predict(X_test_s)                 # надо помнить про scaler!

# === С Pipeline (безопасно!) ===
pipe = Pipeline([
    ('scaler', StandardScaler()),             # шаг 1: масштабирование
    ('classifier', LogisticRegression(max_iter=200)),  # шаг 2: модель
])

# Один fit — обучает ВСЕ шаги
pipe.fit(X_train, y_train)

# Один predict — применяет ВСЕ шаги
y_pred = pipe.predict(X_test)
print(f'Pipeline accuracy: {pipe.score(X_test, y_test):.4f}')

# Доступ к шагам
print(f'Scaler mean: {pipe.named_steps["scaler"].mean_[:2]}')
print(f'Model coef shape: {pipe.named_steps["classifier"].coef_.shape}')

In [ ]:
# Pipeline с большим количеством шагов
from sklearn.decomposition import PCA

pipe_complex = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2)),
    ('classifier', LogisticRegression(max_iter=200)),
])

pipe_complex.fit(X_train, y_train)
print(f'Complex pipeline: {pipe_complex.score(X_test, y_test):.4f}')
print(f'Steps: {[name for name, _ in pipe_complex.steps]}')

<a id='19'></a>
## 19. ColumnTransformer: разные обработки для разных столбцов

Реальные данные содержат числовые и категориальные признаки.
**ColumnTransformer** применяет разные трансформеры к разным столбцам.

Документация: [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html)

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_transformer, make_column_selector

# Создадим "реальные" данные
np.random.seed(42)
n = 200
df = pd.DataFrame({
    'age': np.random.randint(18, 65, n),
    'salary': np.random.normal(60000, 20000, n).astype(int),
    'experience': np.random.randint(0, 30, n),
    'city': np.random.choice(['Moscow', 'SPB', 'Kazan', 'Novosibirsk'], n),
    'education': np.random.choice(['bachelor', 'master', 'phd'], n),
    'is_remote': np.random.choice([0, 1], n),
})
# Добавим пропуски
df.loc[df.sample(20, random_state=42).index, 'salary'] = np.nan
df.loc[df.sample(10, random_state=43).index, 'city'] = np.nan

# Целевая переменная
y = (df['salary'].fillna(60000) > 60000).astype(int)

print(df.head())
print(f'\nПропуски:\n{df.isna().sum()}')

In [ ]:
# Определяем столбцы
num_cols = ['age', 'salary', 'experience']
cat_cols = ['city', 'education']

# Отдельные пайплайны для числовых и категориальных
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')),
])

# Объединяем в ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
    # ('passthrough', 'passthrough', ['is_remote']),  # оставить как есть
], remainder='passthrough')  # остальные столбцы — как есть

# Тестируем
X_processed = preprocessor.fit_transform(df)
print(f'Форма до: {df.shape}')
print(f'Форма после: {X_processed.shape}')
print(f'Имена: {preprocessor.get_feature_names_out()}')

<a id='20'></a>
## 20. Полный Pipeline: preprocessing + model

In [ ]:
# === Полный пайплайн: ColumnTransformer + Модель ===

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),                        # шаг 1: предобработка
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42)),  # шаг 2: модель
])

# Split
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, stratify=y, random_state=42)

# Fit + Predict — работает с СЫРЫМИ данными (DataFrame с пропусками)!
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)

print(f'Train accuracy: {full_pipeline.score(X_train, y_train):.4f}')
print(f'Test accuracy:  {full_pipeline.score(X_test, y_test):.4f}')

# Кросс-валидация
scores = cross_val_score(full_pipeline, df, y, cv=5, scoring='accuracy')
print(f'CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

<a id='21'></a>
## 21. make_pipeline и make_column_transformer

Короткие варианты — имена генерируются автоматически.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector

# make_pipeline — без имён
pipe_short = make_pipeline(
    StandardScaler(),
    PCA(n_components=2),
    LogisticRegression(max_iter=200)
)
print('make_pipeline steps:', [name for name, _ in pipe_short.steps])

# make_column_transformer с автоматическим выбором столбцов
ct_auto = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy='median'), StandardScaler()),
     make_column_selector(dtype_include='number')),   # все числовые
    
    (make_pipeline(SimpleImputer(strategy='most_frequent'),
                   OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')),
     make_column_selector(dtype_include='object')),   # все строковые
    
    remainder='passthrough'
)

print('\nmake_column_transformer:')
print(ct_auto)

---
# Часть VI — Подбор гиперпараметров
---

<a id='22'></a>
## 22. GridSearchCV

Полный перебор всех комбинаций гиперпараметров.

Документация: [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

In [ ]:
from sklearn.model_selection import GridSearchCV

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC()),
])

# Пространство поиска
param_grid = {
    'clf__C': [0.1, 1, 10, 100],
    'clf__kernel': ['linear', 'rbf', 'poly'],
    'clf__gamma': ['scale', 'auto'],
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,                      # 5-fold CV
    scoring='accuracy',
    n_jobs=-1,                 # параллельно
    verbose=0,
    return_train_score=True,
)

grid.fit(X_train, y_train)

print(f'Best params:  {grid.best_params_}')
print(f'Best CV score: {grid.best_score_:.4f}')
print(f'Test score:    {grid.score(X_test, y_test):.4f}')
print(f'Fitted models: {len(grid.cv_results_["mean_test_score"])}')

In [ ]:
# Результаты в DataFrame
results = pd.DataFrame(grid.cv_results_)
cols = ['param_clf__C', 'param_clf__kernel', 'param_clf__gamma',
        'mean_test_score', 'std_test_score', 'rank_test_score']
print(results[cols].sort_values('rank_test_score').head(10))

<a id='23'></a>
## 23. RandomizedSearchCV

Случайный перебор — быстрее при большом пространстве.

Документация: [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint, loguniform

pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42)),
])

# Распределения для параметров
param_distributions = {
    'clf__n_estimators': randint(50, 300),       # случайное целое [50, 300)
    'clf__max_depth': [3, 5, 7, 10, None],       # из списка
    'clf__min_samples_split': randint(2, 20),
    'clf__min_samples_leaf': randint(1, 10),
    'clf__max_features': ['sqrt', 'log2', None],
}

random_search = RandomizedSearchCV(
    pipe_rf,
    param_distributions,
    n_iter=30,                  # число случайных комбинаций
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
)

random_search.fit(X_train, y_train)

print(f'Best params:  {random_search.best_params_}')
print(f'Best CV score: {random_search.best_score_:.4f}')
print(f'Test score:    {random_search.score(X_test, y_test):.4f}')

<a id='24'></a>
## 24. Подбор по полному Pipeline

In [ ]:
# Можно искать параметры ВСЕХ шагов, включая preprocessing!

pipe_full = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('clf', LogisticRegression(max_iter=200)),
])

param_grid = {
    'pca__n_components': [2, 3, 4],             # сколько компонент PCA
    'clf__C': [0.01, 0.1, 1, 10],               # регуляризация LogReg
    'clf__penalty': ['l1', 'l2'],                # тип регуляризации
    'clf__solver': ['liblinear'],                # solver для l1
}

grid = GridSearchCV(pipe_full, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

print(f'Best params: {grid.best_params_}')
print(f'Best CV:     {grid.best_score_:.4f}')
print(f'Test:        {grid.score(X_test, y_test):.4f}')

---
# Часть VII — Продвинутое
---

<a id='25'></a>
## 25. Кастомные трансформеры

Иногда нужна обработка, которой нет в sklearn.

Документация: [Custom transformers](https://scikit-learn.org/stable/developers/develop.html)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class OutlierClipper(BaseEstimator, TransformerMixin):
    """Обрезает выбросы по IQR."""
    
    def __init__(self, factor=1.5):
        self.factor = factor
    
    def fit(self, X, y=None):
        X = np.array(X)
        self.q1_ = np.percentile(X, 25, axis=0)
        self.q3_ = np.percentile(X, 75, axis=0)
        self.iqr_ = self.q3_ - self.q1_
        self.lower_ = self.q1_ - self.factor * self.iqr_
        self.upper_ = self.q3_ + self.factor * self.iqr_
        return self
    
    def transform(self, X):
        X = np.array(X).copy()
        return np.clip(X, self.lower_, self.upper_)

# Использование в Pipeline
pipe_custom = Pipeline([
    ('clipper', OutlierClipper(factor=1.5)),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=200)),
])

X, y = load_iris(return_X_y=True)
scores = cross_val_score(pipe_custom, X, y, cv=5, scoring='accuracy')
print(f'Pipeline с кастомным трансформером: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Трансформер для создания признаков из DataFrame

class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    """Извлекает признаки из datetime столбца."""
    
    def __init__(self, date_column='date'):
        self.date_column = date_column
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        dt = pd.to_datetime(X[self.date_column])
        X['year'] = dt.dt.year
        X['month'] = dt.dt.month
        X['day_of_week'] = dt.dt.dayofweek
        X['is_weekend'] = (dt.dt.dayofweek >= 5).astype(int)
        X = X.drop(columns=[self.date_column])
        return X

# Тест
df_test = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=5),
    'value': [10, 20, 30, 40, 50],
})
print(DateFeatureExtractor('date').fit_transform(df_test))

<a id='26'></a>
## 26. Сохранение и загрузка моделей

Документация: [Model persistence](https://scikit-learn.org/stable/model_persistence.html)

In [ ]:
import joblib
import pickle
import os

# Обучим Pipeline
X, y = load_iris(return_X_y=True)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=200))
pipe.fit(X, y)

# === joblib (рекомендуется для sklearn) ===
joblib.dump(pipe, 'model_pipeline.joblib')
pipe_loaded = joblib.load('model_pipeline.joblib')
print(f'joblib: accuracy = {pipe_loaded.score(X, y):.4f}')

# === pickle ===
with open('model_pipeline.pkl', 'wb') as f:
    pickle.dump(pipe, f)

with open('model_pipeline.pkl', 'rb') as f:
    pipe_pickle = pickle.load(f)
print(f'pickle: accuracy = {pipe_pickle.score(X, y):.4f}')

# Размеры файлов
for f in ['model_pipeline.joblib', 'model_pipeline.pkl']:
    print(f'{f}: {os.path.getsize(f) / 1024:.1f} KB')
    os.remove(f)

print('\nФайлы удалены ✅')

<a id='27'></a>
## 27. Полный пример: от сырых данных до модели

Собираем всё вместе.

In [ ]:
# === STEP 1: Данные ===
np.random.seed(42)
n = 500

raw_data = pd.DataFrame({
    'age': np.random.randint(18, 70, n),
    'income': np.random.lognormal(10.5, 0.8, n).astype(int),
    'experience': np.random.randint(0, 40, n),
    'education': np.random.choice(['high_school', 'bachelor', 'master', 'phd'], n,
                                   p=[0.3, 0.4, 0.2, 0.1]),
    'city': np.random.choice(['Moscow', 'SPB', 'Kazan', 'Novosibirsk', 'Sochi'], n),
    'has_car': np.random.choice([0, 1], n, p=[0.4, 0.6]),
})

# Добавим пропуски
raw_data.loc[raw_data.sample(30, random_state=1).index, 'income'] = np.nan
raw_data.loc[raw_data.sample(20, random_state=2).index, 'education'] = np.nan

# Целевая: "высокий доход"
y = (raw_data['income'].fillna(50000) > 50000).astype(int)

print('Данные:')
print(raw_data.head())
print(f'\nФорма: {raw_data.shape}')
print(f'Пропуски:\n{raw_data.isna().sum()}')
print(f'Баланс классов: {np.bincount(y)}')

In [ ]:
# === STEP 2: Определение столбцов ===
num_features = ['age', 'income', 'experience']
cat_features = ['education', 'city']
pass_features = ['has_car']

# === STEP 3: Preprocessing Pipeline ===
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clipper', OutlierClipper(factor=2.0)),
    ('scaler', StandardScaler()),
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features),
    ('pass', 'passthrough', pass_features),
])

# === STEP 4: Full Pipeline ===
full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42)),
])

print('Pipeline создан:')
print(full_pipe)

In [ ]:
# === STEP 5: Train/Test split ===
X_train, X_test, y_train, y_test = train_test_split(
    raw_data, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# === STEP 6: GridSearch по Pipeline ===
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 10, None],
    'classifier__min_samples_leaf': [1, 3, 5],
}

grid_search = GridSearchCV(
    full_pipe,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0,
)

grid_search.fit(X_train, y_train)

print(f'Best params: {grid_search.best_params_}')
print(f'Best CV F1:  {grid_search.best_score_:.4f}')

In [ ]:
# === STEP 7: Финальная оценка ===
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print('=== Финальные метрики на тесте ===')
print(classification_report(y_test, y_pred, target_names=['Low income', 'High income']))
print(f'ROC AUC: {roc_auc_score(y_test, y_proba):.4f}')

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Low', 'High'],
    cmap='Blues', ax=ax
)
ax.set_title('Итоговая Confusion Matrix')
fig.tight_layout()
plt.show()

In [ ]:
# === STEP 8: Feature importance ===
feature_names_out = best_model.named_steps['preprocessor'].get_feature_names_out()
importances = best_model.named_steps['classifier'].feature_importances_

feat_imp = pd.Series(importances, index=feature_names_out).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Feature Importance (из Pipeline!)')
ax.set_xlabel('Importance')
fig.tight_layout()
plt.show()

<a id='28'></a>
## 28. 📋 Шпаргалка

### Estimator API
```python
estimator.fit(X, y)              # обучить
estimator.predict(X)             # предсказать
estimator.score(X, y)            # оценить
estimator.predict_proba(X)       # вероятности
transformer.fit_transform(X)     # fit + transform
transformer.transform(X)         # только transform (на test!)
```

### Preprocessing
```python
StandardScaler()          # z-score: (x-mean)/std
MinMaxScaler()            # [0, 1]
RobustScaler()            # устойчив к выбросам
OneHotEncoder()           # бинарные столбцы
OrdinalEncoder()          # числа по порядку
SimpleImputer()           # заполнение NaN
KNNImputer()              # заполнение по соседям
PolynomialFeatures()      # полиномы, взаимодействия
PowerTransformer()        # нормализация распределения
```

### Модели — Классификация
```python
LogisticRegression(C, penalty, solver)
KNeighborsClassifier(n_neighbors, weights)
DecisionTreeClassifier(max_depth, min_samples_split)
RandomForestClassifier(n_estimators, max_depth)
GradientBoostingClassifier(n_estimators, learning_rate)
SVC(C, kernel, gamma)
```

### Модели — Регрессия
```python
LinearRegression()
Ridge(alpha)              # L2
Lasso(alpha)              # L1
ElasticNet(alpha, l1_ratio)
KNeighborsRegressor(n_neighbors)
DecisionTreeRegressor(max_depth)
RandomForestRegressor(n_estimators)
GradientBoostingRegressor(n_estimators, learning_rate)
SVR(C, kernel)
```

### Pipeline
```python
# Цепочка
pipe = Pipeline([('name1', step1), ('name2', step2), ...])
pipe = make_pipeline(step1, step2, ...)   # автоимена

# Разные столбцы
ct = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
], remainder='passthrough')

# Полный Pipeline
full = Pipeline([('preprocessor', ct), ('model', clf)])
```

### Подбор параметров
```python
# В param_grid: 'step__param'
GridSearchCV(pipe, param_grid, cv=5, scoring='f1', n_jobs=-1)
RandomizedSearchCV(pipe, distributions, n_iter=50, cv=5)
```

### Валидация
```python
train_test_split(X, y, test_size=0.2, stratify=y)
cross_val_score(model, X, y, cv=5, scoring='accuracy')
cross_validate(model, X, y, cv=5, scoring=[...], return_train_score=True)
```

### Метрики
```python
# Классификация
accuracy_score, precision_score, recall_score, f1_score
roc_auc_score, classification_report, confusion_matrix

# Регрессия
mean_squared_error, mean_absolute_error, r2_score
mean_absolute_percentage_error
```

### Сохранение
```python
joblib.dump(model, 'model.joblib')
model = joblib.load('model.joblib')
```

---

## 🎯 Упражнения для самопроверки

In [ ]:
# Упражнение 1: Построй Pipeline для классификации wine dataset:
# - StandardScaler
# - PCA (подобрать n_components через GridSearch: 2, 5, 10)
# - RandomForest (подобрать n_estimators: 50, 100, 200)
# Оцени лучшую модель на тесте.

X, y = load_wine(return_X_y=True)
# Твой код:


In [ ]:
# Упражнение 2: Создай ColumnTransformer + Pipeline для данных с:
# - числовыми столбцами (impute median + scale)
# - категориальными (impute mode + one-hot)
# Обучи LogisticRegression, выведи classification_report.

from sklearn.datasets import fetch_openml
try:
    titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
    df_t = titanic.data[['age', 'fare', 'sex', 'embarked', 'pclass']].copy()
    y_t = (titanic.target == '1').astype(int)
    print(df_t.head())
    # Твой код:

except Exception as e:
    print(f'Не удалось загрузить: {e}')
    print('Создай свой DataFrame для практики')

In [ ]:
# Упражнение 3: Напиши кастомный трансформер, который:
# - в fit() запоминает медиану каждого столбца
# - в transform() создаёт новые бинарные признаки:
#   1 если значение > медианы, 0 иначе
# Используй его в Pipeline.

# Твой код:


In [ ]:
# Упражнение 4: Сравни 5 моделей на breast_cancer dataset:
# LogReg, KNN, DecisionTree, RandomForest, SVM
# Все в Pipeline со StandardScaler.
# Оцени через 5-fold CV, выведи таблицу: Model | Mean Accuracy | Std

# Твой код:


---

### 🏁 Конец блокнота Scikit-learn

Ты покрыл:
- Estimator API (fit/transform/predict)
- Preprocessing: масштабирование, кодирование, импутация, трансформации
- Отбор признаков
- Основные модели классификации и регрессии
- Метрики (classification + regression)
- **Pipeline** и **ColumnTransformer**
- GridSearchCV / RandomizedSearchCV
- Кастомные трансформеры
- Сохранение моделей
- Полный end-to-end пример

**Все 4 блокнота готовы!** 🎉
- `numpy_tutorial.ipynb`
- `pandas_tutorial.ipynb`
- `visualization_tutorial.ipynb` (Matplotlib + Seaborn)
- `sklearn_tutorial.ipynb`